# Preliminary 실험 전체 실행 — 본 연구에 사용하지 않음

실제자료 기준 탐색, 의도적 유의성 부풀리기, 효과 주입 모의실험, PPML, 개별공연장 좌석 감사·탐색을 각각 새 커널에서 실행하고 종합 보고서를 만든다. 원자료·기존 최종 결과를 수정하지 않는다.

분석 네트워크 요청 없이 실행한다. 실행 중 다른 실험 노트북을 동시에 수정하거나 실행하지 않는다. 원자료의 결측을0원으로 대체하지 않는다. 모든 출력은 test 폴더 안에 둔다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,os,time
import pandas as pd
import nbformat
from nbclient import NotebookClient
from nbconvert import HTMLExporter
from IPython.display import display
ROOT=PROJECT_ROOT
OUT=project_path('jupyter-notebook/sandbox/reports');HTML=OUT/'notebooks';HTML.mkdir(exist_ok=True)
for k in ['OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS']:os.environ[k]='1'
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
plan=project_json_loads((project_path('jupyter-notebook/sandbox/metadata/lab_plan.json')).read_text(encoding='utf-8'))
for p,h in plan['protected_sha256'].items():assert sha(project_path(p))==h,p
names=['01_측정표본_탐색','02_유의성부풀리기_시연','03_효과주입_모의실험','05_PPML_매출모형','06_좌석연결과_핵심상권감사','07_좌석공급_탐색','04_실험결과_보고서']
manifest={'status':'running','started_utc':datetime.now(timezone.utc).isoformat(),'purpose':'PRELIMINARY_ONLY','steps':[],'network_requests':0}
def save(): (OUT/'execution_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
save()

## 순차 실행과 읽기용 출력

잘못된 방법을 의도적으로 시연하는 노트북도 포함된다. 작은 p값만으로 표준오차·표본 선택의 타당성이 확인되는 것은 아니다. 실패가 있으면 숨기지 않고 해당 단계에서 멈춘다.

In [ ]:
for name in names:
    p=project_path('jupyter-notebook/sandbox/notebooks')/(name+'.ipynb');nb=nbformat.read(p,as_version=4)
    source=chr(10).join(c.source for c in nb.cells);started=time.perf_counter()
    print('실행:',name,flush=True)
    try:
        NotebookClient(nb,timeout=1200,kernel_name='python3',resources={'metadata':{'path':str(p.parent)}}).execute()
        nbformat.validate(nb);nbformat.write(nb,p)
        body,_=HTMLExporter(template_name='lab',exclude_input=True,exclude_input_prompt=True,exclude_output_prompt=True).from_notebook_node(nb)
        hp=HTML/(name+'.html');hp.write_text(body,encoding='utf-8')
        manifest['steps'].append({'notebook':p.relative_to(ROOT).as_posix(),'status':'passed','seconds':round(time.perf_counter()-started,2),
            'source_sha256':hashlib.sha256(source.encode()).hexdigest(),'executed_sha256':sha(p),'html':hp.relative_to(ROOT).as_posix()})
        save()
    except Exception as e:
        folder=OUT/'failed';folder.mkdir(exist_ok=True);nbformat.write(nb,folder/p.name)
        manifest.update(status='failed',failed_notebook=name,error_type=type(e).__name__);save();raise
for p,h in plan['protected_sha256'].items():assert sha(project_path(p))==h,p
manifest.update(status='complete',finished_utc=datetime.now(timezone.utc).isoformat(),protected_files_unchanged=len(plan['protected_sha256']))
save();display(pd.DataFrame(manifest['steps'])[['notebook','status','seconds']])

## 결과

종합 보고서: jupyter-notebook/sandbox/reports/report.html. 전체 검정과 실패·미추정은 각 번호의 CSV·summary.json에 저장한다. 이 폴더의 분석과 기준은 본 연구로 자동 반영되지 않는다.